# Thesis Notebook — SRQ1 FINAL COMPARISON

**Owner**: Enrico Manfron
**Institution**: Copenhagen Business School (MSc 2026)
**Created**: 2026-04-24

**Scope**: Aggregate the outputs of all **7 model notebooks** into the headline tables
and figures for Chapter 6 of the thesis.

**Inputs** (7 model notebooks — must be run before this one):

| Notebook | Output dir |
|----------|-----------|
| `thesis_notebook.ipynb` (CSD) | `outputs/` |
| `thesis_notebook_danskvand.ipynb` | `outputs_danskvand/` |
| `thesis_notebook_energidrikke.ipynb` | `outputs_energidrikke/` |
| `thesis_notebook_rtd.ipynb` | `outputs_rtd/` |
| `thesis_notebook_totalbeer.ipynb` | `outputs_totalbeer/` |
| `thesis_notebook_pooled_4.ipynb` | `outputs_pooled_4/` |
| `thesis_notebook_pooled_5.ipynb` | `outputs_pooled_5/` |

**Outputs** (saved to `outputs_final/`):
- `headline_table.csv` — single CSV with all 7 models × 4 metrics
- `pool_vs_spec.csv` — per-category specialized-vs-pooled comparison
- `figures/fig_headline.png` — bar chart of all 7 models
- `figures/fig_pool_vs_spec_heatmap.png` — heatmap pool vs spec deltas
- `chapter6_table.md` — pre-formatted Markdown table for direct paste in thesis

---

# §0 — Setup

**Why**: imports + paths. Define what categories were specialized vs pooled.

In [7]:
from pathlib import Path                                                                                                                                                                                                                                                                           

# Find project root by locating CLAUDE.md -> helps dynamically finding the project root regardless of where the script is run from                                                                                                                                                                                                                                                          
current = Path.cwd()
while current != current.parent:
    if (current / "CLAUDE.md").exists():
        ROOT_DIR = current
        break
    current = current.parent
else:
    raise FileNotFoundError("Could not find project root (CLAUDE.md)")

import sys
print(f"Project root found at: {ROOT_DIR}")
sys.path.insert(0, str(ROOT_DIR))

Project root found at: c:\dev\thesis-manifold


In [8]:
# Import the config file from the project root to centralize all directories
import importlib
import paths
importlib.reload(paths)  # Reload the config module to ensure we have the latest changes

from paths import *

ROOT_DIR = C:\dev\thesis-manifold
THESIS_DIR = C:\dev\thesis-manifold\thesis
THESIS_ANALYSIS_DIR = C:\dev\thesis-manifold\thesis\analysis
THESIS_FIGURES_DIR = C:\dev\thesis-manifold\thesis\analysis\figures
THESIS_NOTEBOOKS_DIR = C:\dev\thesis-manifold\thesis\analysis\notebooks
THESIS_PROMPTS_DIR = C:\dev\thesis-manifold\thesis\analysis\prompts
THESIS_OUTPUTS_DIR = C:\dev\thesis-manifold\thesis\analysis\outputs
FEATURE_MATRIX_OUTPUTS_DIR = C:\dev\thesis-manifold\thesis\analysis\outputs\feature_matrices
CSD_OUTPUTS_DIR = C:\dev\thesis-manifold\thesis\analysis\outputs\csd
DANSKVAND_OUTPUTS_DIR = C:\dev\thesis-manifold\thesis\analysis\outputs\danskvand
ENERGIDRIKKE_OUTPUTS_DIR = C:\dev\thesis-manifold\thesis\analysis\outputs\energidrikke
RTD_V2_OUTPUTS_DIR = C:\dev\thesis-manifold\thesis\analysis\outputs\rtd_v2
TOTALBEER_OUTPUTS_DIR = C:\dev\thesis-manifold\thesis\analysis\outputs\totalbeer
POOLED_4_OUTPUTS_DIR = C:\dev\thesis-manifold\thesis\analysis\outputs\pooled_4
POOLED_5_OUTPUTS_DIR 

In [ ]:
# §0 — Setup
import sys, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# PROJECT_ROOT = Path("/Users/enricomanfron/Desktop/Thesis Maniflod")
ANALYSIS_DIR = ROOT_DIR / "docs" / "thesis" / "analysis"
OUTPUT_DIR   = ANALYSIS_DIR / "outputs_final"
FIGURE_DIR   = OUTPUT_DIR / "figures"
for d in (OUTPUT_DIR, FIGURE_DIR):
    d.mkdir(parents=True, exist_ok=True)

# 5 specialized + 2 pooled = 7 notebooks total
SPECIALIZED = ["csd", "danskvand", "energidrikke", "rtd", "totalbeer"]
POOLED      = ["pooled_4", "pooled_5"]
ALL_NOTEBOOKS = SPECIALIZED + POOLED

# Map notebook → output subdir
def output_dir(name):
    return ANALYSIS_DIR / ("outputs" if name == "csd" else f"outputs_{name}")

print("Existence check:")
for n in ALL_NOTEBOOKS:
    p = output_dir(n) / "all_models_metrics.csv"
    print(f"  {n:<14s} {'✓' if p.exists() else '✗ MISSING'}  {p.relative_to(PROJECT_ROOT)}")


### §0 — Observations + Decisions

- All 7 outputs present? If any ✗, **run that notebook first**.
- Ready to aggregate.

---

# §1 — Headline aggregate table

**Why**: single table with all 7 models × 4 metrics (median MAPE, mean MAPE, WAPE, RMSE).
This is the master table for Chapter 6.

In [ ]:
# §1 — Headline table
rows = []
for n in ALL_NOTEBOOKS:
    p = output_dir(n) / "all_models_metrics.csv"
    if not p.exists():
        print(f"⚠ Skipping {n} (no output)")
        continue
    metrics = pd.read_csv(p)
    for _, r in metrics.iterrows():
        # Identify the model variant (LightGBM_global vs LightGBM_pooled)
        model_name = r["model"]
        rows.append({
            "notebook": n,
            "scope": "specialized" if n in SPECIALIZED else "pooled",
            "model": model_name,
            "mape_median": r.get("mape_median"),
            "mape_mean": r.get("mape_mean"),
            "wape": r.get("wape"),
            "rmse": r.get("rmse"),
        })

headline = pd.DataFrame(rows)
print(f"[Headline table — {len(headline)} model results across {headline['notebook'].nunique()} notebooks]")
print(headline.to_string(index=False, float_format=lambda x: f"{x:,.2f}" if isinstance(x, float) else str(x)))
headline.to_csv(OUTPUT_DIR / "headline_table.csv", index=False)
print(f"\n✅ Saved: {OUTPUT_DIR.name}/headline_table.csv")


### §1 — Observations + Decisions

- _Best LGB across all 7: ..._
- _Best XGB across all 7: ..._
- _Worst (apart from baselines): ..._

---

# §2 — Pool vs Spec — the SRQ1 answer table

**Why**: the central comparison. For each category, compare:
- Specialized model performance (from per-category notebook)
- Pooled-4 performance restricted to that category (only for the 4 new categories)
- Pooled-5 performance restricted to that category

Negative delta = pooling **wins**. Positive delta = specialized **wins**.

In [ ]:
# §2 — Pool vs Spec
def per_brand_mape_filtered(predictions_df, pred_col):
    """Per-brand median MAPE on a subset (e.g. one category)."""
    if len(predictions_df) == 0: return float("nan")
    mapes = []
    for b in predictions_df["brand"].unique():
        sub = predictions_df[predictions_df["brand"] == b]
        yt, yp = sub["sales_units"].values, sub[pred_col].values
        mask = yt > 0
        if mask.sum() == 0: continue
        ape = np.abs((yt[mask] - yp[mask]) / yt[mask]) * 100
        mapes.append(float(np.median(ape)))
    return float(np.median(mapes)) if mapes else float("nan")

# Load specialized predictions per category
spec_preds = {}
for cat in SPECIALIZED:
    p = output_dir(cat) / "predictions_val_all.parquet"
    if p.exists():
        spec_preds[cat] = pd.read_parquet(p)

# Load pooled predictions
pool_preds = {}
for pool in POOLED:
    p = output_dir(pool) / "predictions_val_all.parquet"
    if p.exists():
        pool_preds[pool] = pd.read_parquet(p)

# Build the comparison table
rows = []
for cat in SPECIALIZED:
    if cat not in spec_preds:
        continue
    spec_df = spec_preds[cat]
    spec_lgb_mape = per_brand_mape_filtered(spec_df, "LightGBM_global")
    spec_xgb_mape = per_brand_mape_filtered(spec_df, "XGBoost_global")

    row = {
        "category": cat,
        "n_brands": spec_df["brand"].nunique(),
        "n_rows_val": len(spec_df),
        "spec_LGB": spec_lgb_mape,
        "spec_XGB": spec_xgb_mape,
    }

    for pool in POOLED:
        if pool not in pool_preds:
            row[f"{pool}_LGB"] = np.nan
            row[f"{pool}_XGB"] = np.nan
            continue
        pool_df = pool_preds[pool]
        # Filter pooled predictions to this category only
        if "category" in pool_df.columns:
            pool_cat = pool_df[pool_df["category"] == cat]
        else:
            pool_cat = pool_df  # fallback if category col missing
        row[f"{pool}_LGB"] = per_brand_mape_filtered(pool_cat, "LightGBM_pooled")
        row[f"{pool}_XGB"] = per_brand_mape_filtered(pool_cat, "XGBoost_pooled")

    # Compute deltas (negative = pooling wins for that category)
    if not np.isnan(row.get("pooled_4_LGB", np.nan)):
        row["delta_pool4_LGB_pp"] = row["pooled_4_LGB"] - row["spec_LGB"]
    if not np.isnan(row.get("pooled_5_LGB", np.nan)):
        row["delta_pool5_LGB_pp"] = row["pooled_5_LGB"] - row["spec_LGB"]

    rows.append(row)

cmp_df = pd.DataFrame(rows)
cmp_df.to_csv(OUTPUT_DIR / "pool_vs_spec.csv", index=False)
print("[Pool vs Spec — per-brand median MAPE per category]")
print(cmp_df.to_string(index=False, float_format=lambda x: f"{x:,.2f}" if isinstance(x, float) else str(x)))
print("\nNegative delta = POOLING wins for that category.")
print("Positive delta = SPECIALIZED wins.")


### §2 — Observations + Decisions

**The answer to SRQ1**:
- _For X categories pooling helps (negative delta)._
- _For Y categories specialized wins (positive delta)._
- _Weighted average: pool vs spec overall ..._

---

# §3 — Headline figure 1: bar chart of all 7 models

**Why**: single visual to show the model landscape — baselines (SeasonalNaive, Ridge) at
the top of the bars, then specialized (5 models), then pooled (2 models). Color-coded.

In [ ]:
# §3 — Headline bar chart
sns.set_theme(style="whitegrid", context="paper")

# Filter to LightGBM/XGBoost only (the meaningful models)
plot_df = headline[headline["model"].str.contains("LightGBM|XGBoost", case=False)].copy()
# Order: specialized first (5), then pooled (2), per algo
plot_df["scope_model"] = plot_df["scope"] + " | " + plot_df["model"].str.extract(r"(LightGBM|XGBoost)", expand=False)
plot_df["label"] = plot_df["notebook"] + "\n" + plot_df["model"].str.extract(r"(LightGBM|XGBoost)", expand=False)

fig, ax = plt.subplots(figsize=(11, 5.5))
sns.barplot(
    data=plot_df,
    x="label", y="mape_median",
    hue="scope",
    palette={"specialized": "#3b7ddd", "pooled": "#d97706"},
    ax=ax,
)
ax.set_title("Headline: VAL median MAPE — all 7 model setups (specialized vs pooled, LGB vs XGB)")
ax.set_ylabel("VAL median MAPE (%)")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=20, labelsize=8)
ax.legend(title="Scope", loc="upper right")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "fig_headline.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Saved: {FIGURE_DIR.name}/fig_headline.png")


### §3 — Observations + Decisions

- _Visual takeaway: ..._

---

# §4 — Headline figure 2: pool vs spec heatmap

**Why**: compact heatmap of the per-category deltas. Green cells = pool wins. Red = spec wins.
Goes directly into Chapter 6 as the SRQ1 answer figure.

In [ ]:
# §4 — Heatmap of deltas
delta_cols = [c for c in cmp_df.columns if c.startswith("delta_")]
if delta_cols:
    heat = cmp_df.set_index("category")[delta_cols].T

    fig, ax = plt.subplots(figsize=(8, 3))
    sns.heatmap(
        heat, annot=True, fmt=".2f", cmap="RdYlGn_r", center=0,
        cbar_kws={"label": "Δ MAPE (pp)\n(negative = pool wins)"},
        ax=ax,
    )
    ax.set_title("Pool vs Spec — per-brand MAPE delta (pp)")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "fig_pool_vs_spec_heatmap.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✅ Saved: {FIGURE_DIR.name}/fig_pool_vs_spec_heatmap.png")
else:
    print("(No deltas computed — run pooled notebooks first)")


### §4 — Observations + Decisions

- _Pool wins on N out of 5 categories (LGB)._
- _Average delta: ..._

---

# §5 — Pre-formatted Markdown table for Chapter 6

**Why**: produce a markdown table ready to paste directly into the thesis Chapter 6.
Columns: Category | Spec LGB | Pool-4 LGB | Pool-5 LGB | Δ.

In [ ]:
# §5 — Markdown table for thesis paste
md_lines = ["# Chapter 6 — SRQ1 Headline Table\n"]
md_lines.append("Per-brand median MAPE on validation set (lower is better).\n")
md_lines.append("| Category | n brands | n rows | Spec LGB | Pool-4 LGB | Pool-5 LGB | Δ pool-5 vs spec |")
md_lines.append("|----------|---------:|-------:|---------:|-----------:|-----------:|-----------------:|")

for _, r in cmp_df.iterrows():
    spec   = f"{r['spec_LGB']:.2f}%"   if not pd.isna(r["spec_LGB"])   else "—"
    pool4  = f"{r.get('pooled_4_LGB', np.nan):.2f}%" if not pd.isna(r.get("pooled_4_LGB", np.nan)) else "—"
    pool5  = f"{r.get('pooled_5_LGB', np.nan):.2f}%" if not pd.isna(r.get("pooled_5_LGB", np.nan)) else "—"
    delta5 = r.get("delta_pool5_LGB_pp", np.nan)
    delta_str = f"{delta5:+.2f} pp" if not pd.isna(delta5) else "—"
    md_lines.append(f"| {r['category']} | {r['n_brands']} | {r['n_rows_val']:,} | {spec} | {pool4} | {pool5} | {delta_str} |")

md_lines.append("\n## Interpretation")
md_lines.append("- Negative Δ → pooling improves accuracy for that category (knowledge transfers across categories).")
md_lines.append("- Positive Δ → category-specialized model is better (category patterns are too distinct to share).")
md_lines.append("- The choice of model for production should follow the per-category winner.")

md_text = "\n".join(md_lines)
print(md_text)
(OUTPUT_DIR / "chapter6_table.md").write_text(md_text, encoding="utf-8")
print(f"\n✅ Saved: {OUTPUT_DIR.name}/chapter6_table.md (paste into thesis)")


### §5 — Observations + Decisions

- _Headline message for the thesis: ..._

---

# §6 — Wrap-up

All artefacts saved to `outputs_final/`:
- `headline_table.csv` — full results matrix
- `pool_vs_spec.csv` — per-category deltas
- `figures/fig_headline.png` — bar chart all 7
- `figures/fig_pool_vs_spec_heatmap.png` — delta heatmap
- `chapter6_table.md` — paste-ready markdown table

**Next steps**:
1. Paste `chapter6_table.md` into thesis Chapter 6
2. Use both PNGs as Figure 6.1 + 6.2
3. Update SRQ2 agentic notebook to load the model with the best per-category MAPE